# **10_paired_analysis_for_vit_bg_suppression**

In [2]:
# ============================================================
# 10_paired_analysis_for_vit_bg_suppression
# Standalone paired analysis after runtime disconnect
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/MLP/Project5")
ANALYSIS_OUTPUTS = PROJECT_ROOT / "analysis_outputs"
ANALYSIS_OUTPUTS.mkdir(parents=True, exist_ok=True)

EXP_NAME = "vit_b16_bg_suppression_patch_lambda0001_expand_10ep"
EXP_DIR = PROJECT_ROOT / "final_outputs" / EXP_NAME
EVAL_DIR = EXP_DIR / "eval_jpeg_q30_controlled"

orig_path = EVAL_DIR / "predictions_original.csv"
jpeg_path = EVAL_DIR / "predictions_jpeg_q30_controlled.csv"
summary_path = EVAL_DIR / "summary_jpeg_q30_controlled.csv"

print("orig exists:", orig_path.exists(), orig_path)
print("jpeg exists:", jpeg_path.exists(), jpeg_path)
print("summary exists:", summary_path.exists(), summary_path)

Mounted at /content/drive
orig exists: True /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/eval_jpeg_q30_controlled/predictions_original.csv
jpeg exists: True /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/eval_jpeg_q30_controlled/predictions_jpeg_q30_controlled.csv
summary exists: True /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/eval_jpeg_q30_controlled/summary_jpeg_q30_controlled.csv


# **11_load_predictions**

In [3]:
# ============================================================
# 11_load_predictions
# ============================================================

orig = pd.read_csv(orig_path)
jpeg = pd.read_csv(jpeg_path)

print("orig shape:", orig.shape)
print("jpeg shape:", jpeg.shape)

print("orig columns:")
print(orig.columns.tolist())

display(orig.head())
display(jpeg.head())

summary_df = pd.read_csv(summary_path)
display(summary_df)

orig shape: (12005, 12)
jpeg shape: (12005, 12)
orig columns:
['split', 'image_path', 'filename', 'image_id', 'file_extension', 'true_label_idx', 'true_label_name', 'pred_label_idx', 'pred_label_name', 'correct', 'prob_fake', 'prob_real']


,split,image_path,filename,image_id,file_extension,true_label_idx,true_label_name,pred_label_idx,pred_label_name,correct,prob_fake,prob_real
0,original,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0001.jpg,0001,.jpg,0,fake,1,real,0,0.183774,8.162261e-01
1,original,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0002.jpg,0002,.jpg,0,fake,0,fake,1,1.000000,4.584775e-07
2,original,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0003.jpg,0003,.jpg,0,fake,0,fake,1,0.998565,1.435246e-03
3,original,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0004.jpg,0004,.jpg,0,fake,0,fake,1,0.999294,7.063578e-04
4,original,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0005.jpg,0005,.jpg,0,fake,0,fake,1,1.000000,6.423830e-10


,split,image_path,filename,image_id,file_extension,true_label_idx,true_label_name,pred_label_idx,pred_label_name,correct,prob_fake,prob_real
0,jpeg_q30_controlled,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0001.jpg,0001,.jpg,0,fake,1,real,0,0.012445,9.875550e-01
1,jpeg_q30_controlled,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0002.jpg,0002,.jpg,0,fake,0,fake,1,0.999998,2.183577e-06
2,jpeg_q30_controlled,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0003.jpg,0003,.jpg,0,fake,0,fake,1,0.986691,1.330900e-02
3,jpeg_q30_controlled,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0004.jpg,0004,.jpg,0,fake,0,fake,1,0.975449,2.455069e-02
4,jpeg_q30_controlled,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0005.jpg,0005,.jpg,0,fake,0,fake,1,1.000000,3.697703e-09


,experiment_name,model_name,method_name,lambda_bg,best_epoch,best_val_f1,original_accuracy,original_precision_fake,original_recall_fake,original_f1_fake,...,recall_fake_drop,f1_fake_drop,original_loss,jpeg_loss,original_num_samples,jpeg_num_samples,original_confusion_matrix,jpeg_confusion_matrix,original_eval_time_min,jpeg_eval_time_min
0,vit_b16_bg_suppression_patch_lambda0001_expand...,vit_b16,Patch-level Background Suppression,0.001,5,0.948768,0.944773,0.935816,0.955,0.945311,...,0.03,0.007573,0.213961,0.235202,12005,12005,"[[5730, 270], [393, 5612]]","[[5550, 450], [287, 5718]]",70.355754,101.1811


# **12_build_paired_table**

In [4]:
# ============================================================
# 12_build_paired_table
# Build paired original-vs-JPEG table
# ============================================================

def add_pair_key(df):
    df = df.copy()
    df["pair_key"] = (
        df["true_label_name"].astype(str)
        + "_"
        + df["image_id"].astype(str)
    )
    return df

orig = add_pair_key(orig)
jpeg = add_pair_key(jpeg)

print("Original duplicated pair_key:", orig["pair_key"].duplicated().sum())
print("JPEG duplicated pair_key:", jpeg["pair_key"].duplicated().sum())

orig_small = orig[
    [
        "pair_key",
        "image_path",
        "filename",
        "image_id",
        "file_extension",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]
].copy()

jpeg_small = jpeg[
    [
        "pair_key",
        "image_path",
        "filename",
        "image_id",
        "file_extension",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]
].copy()

paired = orig_small.merge(
    jpeg_small,
    on="pair_key",
    suffixes=("_orig", "_jpeg"),
    how="inner"
)

label_mismatch = (
    paired["true_label_name_orig"] != paired["true_label_name_jpeg"]
).sum()

print("label mismatch:", label_mismatch)
print("paired shape:", paired.shape)

paired["prob_fake_drop"] = paired["prob_fake_orig"] - paired["prob_fake_jpeg"]
paired["prob_fake_change"] = paired["prob_fake_jpeg"] - paired["prob_fake_orig"]
paired["prob_real_change"] = paired["prob_real_jpeg"] - paired["prob_real_orig"]

display(paired.head())

Original duplicated pair_key: 0
JPEG duplicated pair_key: 0
label mismatch: 0
paired shape: (12005, 23)


,pair_key,image_path_orig,filename_orig,image_id_orig,file_extension_orig,true_label_idx_orig,true_label_name_orig,pred_label_idx_orig,pred_label_name_orig,correct_orig,...,true_label_idx_jpeg,true_label_name_jpeg,pred_label_idx_jpeg,pred_label_name_jpeg,correct_jpeg,prob_fake_jpeg,prob_real_jpeg,prob_fake_drop,prob_fake_change,prob_real_change
0,fake_0001,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0001.jpg,0001,.jpg,0,fake,1,real,0,...,0,fake,1,real,0,0.012445,9.875550e-01,0.171329,-0.171329,1.713289e-01
1,fake_0002,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0002.jpg,0002,.jpg,0,fake,0,fake,1,...,0,fake,0,fake,1,0.999998,2.183577e-06,0.000002,-0.000002,1.725099e-06
2,fake_0003,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0003.jpg,0003,.jpg,0,fake,0,fake,1,...,0,fake,0,fake,1,0.986691,1.330900e-02,0.011874,-0.011874,1.187375e-02
3,fake_0004,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0004.jpg,0004,.jpg,0,fake,0,fake,1,...,0,fake,0,fake,1,0.975449,2.455069e-02,0.023844,-0.023844,2.384433e-02
4,fake_0005,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0005.jpg,0005,.jpg,0,fake,0,fake,1,...,0,fake,0,fake,1,1.000000,3.697703e-09,0.000000,0.000000,3.055320e-09


# **13_transition_analysis**

In [5]:
# ============================================================
# 13_transition_analysis
# Classify original -> JPEG prediction transitions
# ============================================================

def classify_transition(row):
    true_label = row["true_label_name_orig"]
    orig_pred = row["pred_label_name_orig"]
    jpeg_pred = row["pred_label_name_jpeg"]

    if true_label == "fake":
        if orig_pred == "fake" and jpeg_pred == "fake":
            return "fake_TP_to_TP"
        elif orig_pred == "fake" and jpeg_pred == "real":
            return "fake_TP_to_FN"
        elif orig_pred == "real" and jpeg_pred == "fake":
            return "fake_FN_to_TP"
        elif orig_pred == "real" and jpeg_pred == "real":
            return "fake_FN_to_FN"

    if true_label == "real":
        if orig_pred == "real" and jpeg_pred == "real":
            return "real_TN_to_TN"
        elif orig_pred == "real" and jpeg_pred == "fake":
            return "real_TN_to_FP"
        elif orig_pred == "fake" and jpeg_pred == "real":
            return "real_FP_to_TN"
        elif orig_pred == "fake" and jpeg_pred == "fake":
            return "real_FP_to_FP"

    return "other"

paired["transition"] = paired.apply(classify_transition, axis=1)

transition_counts = paired["transition"].value_counts().reset_index()
transition_counts.columns = ["transition", "count"]

display(transition_counts)

paired_save_path = ANALYSIS_OUTPUTS / "vit_b16_bg_suppression_patch_lambda0001_paired_predictions_original_vs_jpeg.csv"
paired.to_csv(paired_save_path, index=False, encoding="utf-8-sig")

print("Saved paired predictions:", paired_save_path)

,transition,count
0,real_TN_to_TN,5606
1,fake_TP_to_TP,5547
2,real_FP_to_FP,281
3,fake_FN_to_FN,267
4,fake_TP_to_FN,183
5,real_FP_to_TN,112
6,real_TN_to_FP,6
7,fake_FN_to_TP,3


Saved paired predictions: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_b16_bg_suppression_patch_lambda0001_paired_predictions_original_vs_jpeg.csv


# **14_transition_summary**

In [6]:
# ============================================================
# 14_transition_summary
# Save compact transition summary
# ============================================================

fake_subset = paired[paired["true_label_name_orig"] == "fake"].copy()
real_subset = paired[paired["true_label_name_orig"] == "real"].copy()

count_dict = paired["transition"].value_counts().to_dict()

bg_transition_summary = pd.DataFrame([
    {
        "model_key": "vit_b16_bg_suppression_patch_lambda0001",
        "model_name": "ViT-B/16 + Patch-BG-Suppression",
        "method_name": "Patch-level Background Suppression",
        "lambda_bg": 0.001,
        "fake_total": len(fake_subset),
        "real_total": len(real_subset),

        "fake_TP_to_TP": count_dict.get("fake_TP_to_TP", 0),
        "fake_TP_to_FN": count_dict.get("fake_TP_to_FN", 0),
        "fake_FN_to_TP": count_dict.get("fake_FN_to_TP", 0),
        "fake_FN_to_FN": count_dict.get("fake_FN_to_FN", 0),

        "real_TN_to_TN": count_dict.get("real_TN_to_TN", 0),
        "real_TN_to_FP": count_dict.get("real_TN_to_FP", 0),
        "real_FP_to_TN": count_dict.get("real_FP_to_TN", 0),
        "real_FP_to_FP": count_dict.get("real_FP_to_FP", 0),

        "fake_TP_to_FN_ratio_percent": count_dict.get("fake_TP_to_FN", 0) / len(fake_subset) * 100,

        "mean_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].mean(),
        "median_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].median(),
        "std_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].std(),

        "mean_prob_fake_drop_real_images": real_subset["prob_fake_drop"].mean(),
        "median_prob_fake_drop_real_images": real_subset["prob_fake_drop"].median(),
        "std_prob_fake_drop_real_images": real_subset["prob_fake_drop"].std(),
    }
])

summary_save_path = ANALYSIS_OUTPUTS / "vit_b16_bg_suppression_patch_lambda0001_transition_summary_original_vs_jpeg.csv"
bg_transition_summary.to_csv(summary_save_path, index=False, encoding="utf-8-sig")

print("Saved:", summary_save_path)
display(bg_transition_summary)

Saved: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_b16_bg_suppression_patch_lambda0001_transition_summary_original_vs_jpeg.csv


,model_key,model_name,method_name,lambda_bg,fake_total,real_total,fake_TP_to_TP,fake_TP_to_FN,fake_FN_to_TP,fake_FN_to_FN,...,real_TN_to_FP,real_FP_to_TN,real_FP_to_FP,fake_TP_to_FN_ratio_percent,mean_prob_fake_drop_fake_images,median_prob_fake_drop_fake_images,std_prob_fake_drop_fake_images,mean_prob_fake_drop_real_images,median_prob_fake_drop_real_images,std_prob_fake_drop_real_images
0,vit_b16_bg_suppression_patch_lambda0001,ViT-B/16 + Patch-BG-Suppression,Patch-level Background Suppression,0.001,6000,6005,5547,183,3,267,...,6,112,281,3.05,0.030519,0.000003,0.110376,0.017235,0.000007,0.07834


# **15_compare_vit_three_methods**

In [7]:
# ============================================================
# 15_compare_vit_three_methods
# Compare ViT baseline vs LazyStrike vs Patch-BG-Suppression
# ============================================================

# Known baseline / LazyStrike values from previous analyses
vit_three_compare = pd.DataFrame([
    {
        "model_key": "vit_b16",
        "model_name": "ViT-B/16",
        "method_name": "Baseline",
        "fake_total": 6000,
        "fake_TP_to_FN": 102,
        "fake_FN_to_TP": 19,
        "fake_TP_to_FN_ratio_percent": 102 / 6000 * 100,
        "mean_prob_fake_drop_fake_images": 0.016788,
    },
    {
        "model_key": "vit_b16_lazystrike_k1",
        "model_name": "ViT-B/16 + LazyStrike-k1",
        "method_name": "LazyStrike-k1",
        "fake_total": 6000,
        "fake_TP_to_FN": 136,
        "fake_FN_to_TP": 12,
        "fake_TP_to_FN_ratio_percent": 136 / 6000 * 100,
        "mean_prob_fake_drop_fake_images": 0.019980,
    },
    {
        "model_key": "vit_b16_bg_suppression_patch_lambda0001",
        "model_name": "ViT-B/16 + Patch-BG-Suppression",
        "method_name": "Patch-level Background Suppression",
        "fake_total": int(bg_transition_summary.loc[0, "fake_total"]),
        "fake_TP_to_FN": int(bg_transition_summary.loc[0, "fake_TP_to_FN"]),
        "fake_FN_to_TP": int(bg_transition_summary.loc[0, "fake_FN_to_TP"]),
        "fake_TP_to_FN_ratio_percent": float(bg_transition_summary.loc[0, "fake_TP_to_FN_ratio_percent"]),
        "mean_prob_fake_drop_fake_images": float(bg_transition_summary.loc[0, "mean_prob_fake_drop_fake_images"]),
    },
])

save_path = ANALYSIS_OUTPUTS / "vit_baseline_vs_lazystrike_vs_bg_suppression_paired_comparison.csv"
vit_three_compare.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
display(vit_three_compare)

Saved: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_baseline_vs_lazystrike_vs_bg_suppression_paired_comparison.csv


,model_key,model_name,method_name,fake_total,fake_TP_to_FN,fake_FN_to_TP,fake_TP_to_FN_ratio_percent,mean_prob_fake_drop_fake_images
0,vit_b16,ViT-B/16,Baseline,6000,102,19,1.700000,0.016788
1,vit_b16_lazystrike_k1,ViT-B/16 + LazyStrike-k1,LazyStrike-k1,6000,136,12,2.266667,0.019980
2,vit_b16_bg_suppression_patch_lambda0001,ViT-B/16 + Patch-BG-Suppression,Patch-level Background Suppression,6000,183,3,3.050000,0.030519


# **16_compare_summary_metrics**

In [8]:
# ============================================================
# 16_compare_summary_metrics
# Compare original/JPEG metrics for ViT baseline, LazyStrike, and BG suppression
# ============================================================

bg_summary = pd.read_csv(summary_path).iloc[0].to_dict()

vit_summary_compare = pd.DataFrame([
    {
        "model_name": "ViT-B/16",
        "method_name": "Baseline",
        "best_epoch": None,
        "best_val_f1": None,
        "original_accuracy": 0.9471,
        "original_recall_fake": 0.9378,
        "original_f1_fake": 0.9466,
        "jpeg_accuracy": 0.9425,
        "jpeg_recall_fake": 0.9240,
        "jpeg_f1_fake": 0.9428,
        "recall_fake_drop": 0.0138,
        "f1_fake_drop": 0.0038,
    },
    {
        "model_name": "ViT-B/16 + LazyStrike-k1",
        "method_name": "LazyStrike-k1",
        "best_epoch": 7,
        "best_val_f1": 0.949902,
        "original_accuracy": 0.942441,
        "original_recall_fake": 0.956333,
        "original_f1_fake": 0.943207,
        "jpeg_accuracy": 0.939608,
        "jpeg_recall_fake": 0.935667,
        "jpeg_f1_fake": 0.939346,
        "recall_fake_drop": 0.020667,
        "f1_fake_drop": 0.003861,
    },
    {
        "model_name": "ViT-B/16 + Patch-BG-Suppression",
        "method_name": "Patch-level Background Suppression",
        "best_epoch": bg_summary["best_epoch"],
        "best_val_f1": bg_summary["best_val_f1"],
        "original_accuracy": bg_summary["original_accuracy"],
        "original_recall_fake": bg_summary["original_recall_fake"],
        "original_f1_fake": bg_summary["original_f1_fake"],
        "jpeg_accuracy": bg_summary["jpeg_accuracy"],
        "jpeg_recall_fake": bg_summary["jpeg_recall_fake"],
        "jpeg_f1_fake": bg_summary["jpeg_f1_fake"],
        "recall_fake_drop": bg_summary["recall_fake_drop"],
        "f1_fake_drop": bg_summary["f1_fake_drop"],
    },
])

save_path = ANALYSIS_OUTPUTS / "vit_baseline_vs_lazystrike_vs_bg_suppression_summary_metrics.csv"
vit_summary_compare.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
display(vit_summary_compare)

Saved: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_baseline_vs_lazystrike_vs_bg_suppression_summary_metrics.csv


,model_name,method_name,best_epoch,best_val_f1,original_accuracy,original_recall_fake,original_f1_fake,jpeg_accuracy,jpeg_recall_fake,jpeg_f1_fake,recall_fake_drop,f1_fake_drop
0,ViT-B/16,Baseline,NaN,NaN,0.947100,0.937800,0.946600,0.942500,0.924000,0.942800,0.013800,0.003800
1,ViT-B/16 + LazyStrike-k1,LazyStrike-k1,7.0,0.949902,0.942441,0.956333,0.943207,0.939608,0.935667,0.939346,0.020667,0.003861
2,ViT-B/16 + Patch-BG-Suppression,Patch-level Background Suppression,5.0,0.948768,0.944773,0.955000,0.945311,0.938609,0.925000,0.937738,0.030000,0.007573
